# LeetCode 3495 — Minimum Operations to Make Array Elements Zero bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-make-array-elements-zero/
- Độ khó: **Hard**
- Chủ đề: **Đếm theo lũy thừa của 4** (số lần chia cho 4 tới 0), **ghép cặp công việc** (cận dưới `⌈S/2⌉` và `max`), tổng trên khoảng bằng **nhóm theo độ dài**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minOperations(self, queries: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Tính số lần chia `⌊x/4⌋` để `x` về 0: `d(x) = ⌊log4 x⌋ + 1` (số chữ số của `x` ở cơ số 4).
2. Biến bài toán thành **xếp lịch**: mỗi phần tử cần `d(x)` "đơn vị việc", mỗi thao tác xử lý tối đa **2 đơn vị của 2 phần tử khác nhau** ⇒ đáp án `max(⌈S/2⌉, max d)`.
3. Chứng minh cận dưới và chỉ ra cách đạt (luôn ghép hai phần tử còn nhiều việc nhất).
4. Tính `S = Σ_{x=l}^{r} d(x)` trong `O(log r)` bằng cách **nhóm** các số có cùng `d` (đoạn `[4^{k−1}, 4^k − 1]`).
5. Kiểm chứng bằng BFS vét cạn trên mảng nhỏ.

## 2. Đề bài & ví dụ

Mỗi truy vấn `[l, r]` tạo mảng `nums = [l, l+1, …, r]`. Một thao tác: chọn **hai** phần tử `a`, `b` và thay bằng `⌊a/4⌋`, `⌊b/4⌋`. Với mỗi truy vấn tìm số thao tác ít nhất để mọi phần tử về 0; trả về **tổng** các đáp án.

**Ví dụ 1:** `queries = [[1,2],[2,4]]` → `3` (`1 + 2`).

**Ví dụ 2:** `queries = [[2,6]]` → `4`.

**Ràng buộc:** `1 ≤ q ≤ 10^5`, `1 ≤ l < r ≤ 10^9`.

In [1]:
from typing import List
examples = [([[1, 2], [2, 4]], 3), ([[2, 6]], 4)]
for q, exp in examples:
    print(q, '-> kỳ vọng', exp)

[[1, 2], [2, 4]] -> kỳ vọng 3
[[2, 6]] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Số lần chia cho 4 tới 0

In [2]:
def d(x):
    c = 0
    while x:
        x //= 4
        c += 1
    return c

print([(x, d(x)) for x in [1, 3, 4, 15, 16, 63, 64]])

[(1, 1), (3, 1), (4, 2), (15, 2), (16, 3), (63, 3), (64, 4)]


`d(x) = k` ⇔ `4^{k−1} ≤ x ≤ 4^k − 1`.

Một công thức số nguyên tiện lợi: số chữ số cơ số 4 của `x` bằng `⌈(số bit của x) / 2⌉` — vì mỗi chữ số cơ số 4 ứng với đúng 2 bit. Trong Python: `d(x) = (x.bit_length() + 1) // 2`. Cách này `O(1)`, không có sai số, và cho thấy `d(x)` chỉ nhận khoảng 16 giá trị khi `x ≤ 10^9`.

In [3]:
def d_bits(x):
    return (x.bit_length() + 1) // 2

assert all(d_bits(x) == d(x) for x in range(1, 100000))
print([(x, bin(x), d_bits(x)) for x in [1, 3, 4, 15, 16, 63, 64]])
print('d(10^9) =', d_bits(10**9), '-> chỉ cần xét k = 1..', d_bits(10**9))

[(1, '0b1', 1), (3, '0b11', 1), (4, '0b100', 2), (15, '0b1111', 2), (16, '0b10000', 3), (63, '0b111111', 3), (64, '0b1000000', 4)]
d(10^9) = 15 -> chỉ cần xét k = 1.. 15


Bảng nhỏ các nhóm đầu tiên (giúp hình dung bước "nhóm theo lũy thừa 4" ở mục 7.3):

| k | đoạn `[4^{k−1}, 4^k − 1]` | số phần tử | đóng góp mỗi số |
|---|---|---|---|
| 1 | `[1, 3]` | 3 | 1 |
| 2 | `[4, 15]` | 12 | 2 |
| 3 | `[16, 63]` | 48 | 3 |
| 4 | `[64, 255]` | 192 | 4 |

### 3.2. Cận dưới khi mỗi bước xử lý hai việc

Tổng việc `S = Σ d(x)`; mỗi thao tác giảm tổng tối đa 2 ⇒ cần `≥ ⌈S/2⌉`. Một phần tử cần `d(x)` thao tác **riêng** (mỗi thao tác chỉ chia nó một lần) ⇒ cần `≥ max d`.

### 3.3. Đạt cận: ghép hai phần tử còn nhiều việc nhất

Nếu `max d ≤ ⌈S/2⌉`, lặp "chọn hai phần tử có việc còn lại lớn nhất" sẽ hết việc sau đúng `⌈S/2⌉` bước (kỹ thuật cổ điển, giống "ghép cặp task khác loại"). Thao tác được phép chọn phần tử đã bằng 0 (chia 0 vẫn 0) nên bước cuối lẻ không sao.

In [4]:
import heapq
def greedy_pairs(work):
    h = [-w for w in work if w]
    heapq.heapify(h)
    ops = 0
    while h:
        a = -heapq.heappop(h)
        b = -heapq.heappop(h) if h else 0
        ops += 1
        if a - 1: heapq.heappush(h, -(a - 1))
        if b - 1 > 0: heapq.heappush(h, -(b - 1))
    return ops

work = [d(x) for x in range(2, 7)]
print('việc:', work, '| tham lam:', greedy_pairs(work), '| công thức:', max((sum(work) + 1) // 2, max(work)))

việc: [1, 1, 2, 2, 2] | tham lam: 4 | công thức: 4


### 3.4. Tổng `d(x)` trên khoảng bằng nhóm

Với mỗi `k = 1, 2, …`: các `x ∈ [max(l, 4^{k−1}), min(r, 4^k − 1)]` đóng góp `k` mỗi số. Chỉ có ~16 nhóm cho `r ≤ 10^9`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `[2, 3, 4, 5, 6]` cần việc `[1, 1, 2, 2, 2]`, `S = 8`, `max = 2` ⇒ `⌈8/2⌉ = 4` ✓.

Ví dụ 1, truy vấn `[2,4]`: việc `[1, 1, 2]`, `S = 4` ⇒ 2 ✓; truy vấn `[1, 2]`: `[1, 1]` ⇒ 1 ✓.

Vì mảng gồm các số **liên tiếp** với `l < r`, `max d = d(r)` và `S − d(r) ≥ d(r − 1) ≥ d(r) − 1` ⇒ `⌈S/2⌉ ≥ d(r)` — tức đáp án luôn là `⌈S/2⌉` (nhưng viết `max` cho an toàn).

## 5. Cách 1 — Brute Force: BFS trên trạng thái mảng

In [5]:
from collections import deque

def min_ops_bfs(nums):
    start = tuple(sorted(nums))
    if not any(start):
        return 0
    dist = {start: 0}
    dq = deque([start])
    n = len(start)
    while dq:
        st = dq.popleft()
        for i in range(n):
            for j in range(i + 1, n):
                nx = list(st)
                nx[i] //= 4; nx[j] //= 4
                nx = tuple(sorted(nx))
                if nx not in dist:
                    dist[nx] = dist[st] + 1
                    if not any(nx):
                        return dist[nx]
                    dq.append(nx)

def answer_brute(queries):
    return sum(min_ops_bfs(list(range(l, r + 1))) for l, r in queries)

for q, exp in examples:
    print(answer_brute(q), '| kỳ vọng', exp)

3 | kỳ vọng 3
4 | kỳ vọng 4


**Độ phức tạp:** số trạng thái bùng nổ — chỉ với mảng rất ngắn, giá trị nhỏ.

**Nút thắt:** chỉ "số việc còn lại" của mỗi phần tử là quan trọng, không phải giá trị cụ thể.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề:** với danh sách việc `w_1..w_m` (`m ≥ 2`), số thao tác tối thiểu là `max(⌈S/2⌉, max w)`.

- Cận dưới: mục 3.2.
- Đạt được: luôn chọn hai phần tử có việc còn lại lớn nhất; nếu `max w ≤ S − max w` thì sau mỗi bước điều kiện vẫn giữ (hoặc `S` còn ≤ 1), nên mỗi bước (trừ có thể bước cuối) làm đúng 2 đơn vị việc.

**Tính `S` nhanh:** cộng theo nhóm `d = k` trên đoạn `[l, r]`.

Đáp án tổng = `Σ_queries max(⌈S/2⌉, d(r))`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam bằng heap trên danh sách việc (mảng nhỏ)

In [6]:
def answer_greedy(queries):
    return sum(greedy_pairs([d(x) for x in range(l, r + 1)]) for l, r in queries)

for q, exp in examples:
    print(answer_greedy(q), '| kỳ vọng', exp)

3 | kỳ vọng 3
4 | kỳ vọng 4


**Độ phức tạp:** `O(S log n)` mỗi truy vấn.

### 7.2. Cách 3 — Công thức, tính `S` bằng duyệt từng số (`O(r − l)`)

In [7]:
def answer_linear(queries):
    total = 0
    for l, r in queries:
        S = sum(d(x) for x in range(l, r + 1))
        total += max((S + 1) // 2, d(r))
    return total

for q, exp in examples:
    print(answer_linear(q), '| kỳ vọng', exp)

3 | kỳ vọng 3
4 | kỳ vọng 4


### 7.3. Cách 4 — Công thức + nhóm theo lũy thừa 4 (lời giải chính)

In [8]:
def work_sum(l, r):
    S, k, lo = 0, 1, 1
    while lo <= r:
        hi = lo * 4 - 1
        a, b = max(l, lo), min(r, hi)
        if a <= b:
            S += k * (b - a + 1)
        lo *= 4
        k += 1
    return S

def answer_groups(queries):
    return sum(max((work_sum(l, r) + 1) // 2, d(r)) for l, r in queries)

for q, exp in examples:
    print(answer_groups(q), '| kỳ vọng', exp)

3 | kỳ vọng 3
4 | kỳ vọng 4


**Độ phức tạp:** `O(log_4 r)` ≈ 16 bước mỗi truy vấn.

### 7.4. Cách 5 — Tổng tiền tố `F(n) = Σ_{x ≤ n} d(x)`

`S(l, r) = F(r) − F(l − 1)`, với `F` tính theo nhóm — tiện khi cần nhiều truy vấn trên cùng hàm.

In [9]:
def F(n):
    S, k, lo = 0, 1, 1
    while lo <= n:
        S += k * (min(n, lo * 4 - 1) - lo + 1)
        lo *= 4
        k += 1
    return S

print(F(6) - F(1), work_sum(2, 6))

8 8


## 8. TRACE từng bước

Trace cách 4 cho truy vấn `[2, 70]`: in từng nhóm `d = k` và tổng.

In [10]:
def trace(l, r):
    S, k, lo = 0, 1, 1
    while lo <= r:
        hi = lo * 4 - 1
        a, b = max(l, lo), min(r, hi)
        if a <= b:
            S += k * (b - a + 1)
            print(f"  nhóm d={k}: [{lo}, {hi}] ∩ [{l}, {r}] = [{a}, {b}] -> {b - a + 1} số × {k} = {k * (b - a + 1)}")
        lo *= 4; k += 1
    ans = max((S + 1) // 2, d(r))
    print(f"S = {S}, max d = {d(r)} -> đáp án = {ans}")
    return ans

trace(2, 70)
print('tham lam heap:', greedy_pairs([d(x) for x in range(2, 71)]))

  nhóm d=1: [1, 3] ∩ [2, 70] = [2, 3] -> 2 số × 1 = 2
  nhóm d=2: [4, 15] ∩ [2, 70] = [4, 15] -> 12 số × 2 = 24
  nhóm d=3: [16, 63] ∩ [2, 70] = [16, 63] -> 48 số × 3 = 144
  nhóm d=4: [64, 255] ∩ [2, 70] = [64, 70] -> 7 số × 4 = 28
S = 198, max d = 4 -> đáp án = 99
tham lam heap: 99


### 8.2. Trace chiến lược ghép cặp (minh họa bổ đề)

Với ví dụ 2 (`[2, 3, 4, 5, 6]`, việc `[1, 1, 2, 2, 2]`), in từng thao tác: hai phần tử được chọn và việc còn lại. Thấy rõ mỗi bước làm đúng 2 đơn vị việc cho tới khi hết.

In [11]:
def trace_pairs(nums):
    work = {x: d(x) for x in nums}
    step = 0
    while any(work.values()):
        order = sorted((w, x) for x, w in work.items() if w)[::-1]
        pick = [x for _, x in order[:2]]
        step += 1
        for x in pick:
            work[x] -= 1
        print(f"thao tác {step}: chọn {pick} -> việc còn lại {dict(work)}")
    return step

print('tổng số thao tác:', trace_pairs([2, 3, 4, 5, 6]))
print('công thức:', max((work_sum(2, 6) + 1) // 2, d(6)))

thao tác 1: chọn [6, 5] -> việc còn lại {2: 1, 3: 1, 4: 2, 5: 1, 6: 1}
thao tác 2: chọn [4, 6] -> việc còn lại {2: 1, 3: 1, 4: 1, 5: 1, 6: 0}
thao tác 3: chọn [5, 4] -> việc còn lại {2: 1, 3: 1, 4: 0, 5: 0, 6: 0}
thao tác 4: chọn [3, 2] -> việc còn lại {2: 0, 3: 0, 4: 0, 5: 0, 6: 0}
tổng số thao tác: 4
công thức: 4


Ở thao tác cuối, hai phần tử cuối cùng còn việc được xử lý cùng lúc. Nếu tổng việc lẻ thì thao tác cuối chỉ có một phần tử còn việc (phần tử kia là một số 0 — được phép chọn).

Ta cũng có thể nhìn bổ đề qua trường hợp điều kiện `max ≤ S − max` **không** thỏa: mảng `[1, 4^10]` cần việc `[1, 11]`; `⌈12/2⌉ = 6` nhưng phần tử lớn cần 11 lần riêng ⇒ đáp án 11 = `max`. Với các số **liên tiếp** điều này không xảy ra (mục 4).

In [12]:
w = [d(1), d(4 ** 10)]
print('việc:', w, '| ⌈S/2⌉ =', (sum(w) + 1) // 2, '| max =', max(w), '| tham lam:', greedy_pairs(w))

việc: [1, 11] | ⌈S/2⌉ = 6 | max = 11 | tham lam: 11


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def minOperations(self, queries: List[List[int]]) -> int:
        total = 0
        for l, r in queries:
            S, k, lo = 0, 1, 1
            while lo <= r:
                hi = lo * 4 - 1
                a = l if l > lo else lo
                b = r if r < hi else hi
                if a <= b:
                    S += k * (b - a + 1)
                lo *= 4
                k += 1
            dr = k - 1                      # d(r): nhóm cuối cùng chứa r
            total += max((S + 1) // 2, dr)
        return total

for q, exp in examples:
    got = Solution().minOperations(q)
    print(got, 'OK' if got == exp else 'SAI')

3 OK
4 OK


Sau vòng lặp, `k − 1` là chỉ số nhóm cuối có `lo ≤ r`, tức `d(r)`.

## 10. Vì sao thuật toán đúng

1. **Việc của mỗi phần tử:** phần tử `x` về 0 sau đúng `d(x)` lần bị chọn (mỗi lần chia 4 bỏ một chữ số cơ số 4). Chọn phần tử đã bằng 0 không có tác dụng nhưng hợp lệ.
2. **Cận dưới:** mỗi thao tác giảm tổng việc còn lại tối đa 2 và giảm việc của mỗi phần tử tối đa 1 ⇒ `≥ max(⌈S/2⌉, max d)`.
3. **Đạt được:** chiến lược "hai phần tử còn nhiều việc nhất" giữ bất biến `max ≤ ⌈S/2⌉` (khi `max` bằng tổng phần còn lại thì ghép phần tử lớn nhất với mọi phần tử khác lần lượt); mỗi bước làm 2 đơn vị cho tới khi `S ≤ 1`.
4. **Nhóm:** `d(x) = k` ⇔ `x ∈ [4^{k−1}, 4^k − 1]` nên tổng theo nhóm đúng bằng `S`.
5. **Vì sao với số liên tiếp luôn có `⌈S/2⌉ ≥ d(r)`:** `r − 1 ≥ l` có trong mảng và `d(r − 1) ≥ d(r) − 1` (hai số liền nhau khác nhau tối đa một chữ số cơ số 4). Do đó `S ≥ d(r) + d(r−1) ≥ 2d(r) − 1` ⇒ `⌈S/2⌉ ≥ d(r)`. Bài tập 5 kiểm chứng điều này bằng máy.
6. **Bất biến của tham lam (chi tiết):** gọi `M` là việc lớn nhất, `R = S − M`. Khi `M ≤ R` và `S ≥ 2`, luôn có ít nhất hai phần tử còn việc, nên thao tác làm đúng 2 đơn vị. Sau khi giảm hai phần tử lớn nhất đi 1, nếu phần tử lớn nhất cũ vẫn lớn nhất thì `M − 1 ≤ R − 1`; nếu một phần tử khác bằng nó thì hai giá trị lớn nhất bằng nhau nên `M ≤ R` hiển nhiên. Vậy bất biến được giữ tới khi `S ≤ 1`, tổng số thao tác là `⌈S/2⌉`.

## 11. Phân tích Time / Space Complexity

| Cách | Mỗi truy vấn | Space |
|---|---|---|
| 1. BFS | mũ | mũ |
| 2. Heap tham lam | `O(S log n)` | `O(n)` |
| 3. Công thức, duyệt số | `O(r − l)` | `O(1)` |
| 4. Công thức + nhóm | `O(log r)` | `O(1)` |
| 5. Tổng tiền tố `F` | `O(log r)` | `O(1)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [14]:
impls = {'greedy': answer_greedy, 'linear': answer_linear, 'groups': answer_groups,
         'submit': lambda q: Solution().minOperations(q)}
edge = [([[1, 2]], 1), ([[3, 4]], 2), ([[1, 3]], 2), ([[15, 16]], 3)]
for q, exp in examples + edge:
    assert answer_brute(q) == exp, q
    for name, f in impls.items():
        assert f(q) == exp, (name, q)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với BFS (mảng ngắn) và với tham lam (mảng dài hơn)

In [15]:
import random
random.seed(3495)
cnt = 0
for _ in range(60):
    l = random.randint(1, 30)
    r = l + random.randint(1, 4)
    q = [[l, r]]
    assert answer_groups(q) == answer_brute(q), q
    cnt += 1
for _ in range(300):
    l = random.randint(1, 5000)
    r = l + random.randint(1, 300)
    assert answer_groups([[l, r]]) == answer_greedy([[l, r]]) == answer_linear([[l, r]])
    cnt += 1
print('Khớp trên', cnt, 'truy vấn ngẫu nhiên.')

Khớp trên

 360 truy vấn ngẫu nhiên.


### 12.3. Ca lớn (`10^5` truy vấn, `r` tới `10^9`)

In [16]:
import time
random.seed(5)
qs = []
for _ in range(10**5):
    l = random.randint(1, 10**9 - 1)
    qs.append([l, random.randint(l + 1, 10**9)])
t = time.time(); ans = Solution().minOperations(qs); t1 = time.time() - t
assert ans == answer_groups(qs)
print(f"{ans} | {t1:.2f}s")

186268621322578 | 0.14s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng `d(x)` mà không chia đôi (coi mỗi thao tác chỉ xử lý một phần tử)

In [17]:
def wrong_no_half(queries):
    return sum(work_sum(l, r) for l, r in queries)

print('sai :', wrong_no_half(examples[1][0]), '| đúng:', examples[1][1])

sai : 8 | đúng: 4


**Vì sao sai:** mỗi thao tác chia **hai** phần tử cùng lúc.

### Lỗi 2 — Dùng `⌊S/2⌋` thay vì `⌈S/2⌉`

In [18]:
def wrong_floor(queries):
    return sum(work_sum(l, r) // 2 for l, r in queries)

print('sai :', wrong_floor([[1, 3]]), '| đúng:', Solution().minOperations([[1, 3]]))

sai : 1 | đúng: 2


**Vì sao sai:** tổng việc lẻ cần thêm một thao tác cho đơn vị cuối cùng.

### Lỗi 3 — Tính `d(x)` bằng `math.log(x, 4)` (sai số dấu phẩy động)

In [19]:
import math
def d_log(x):
    return int(math.log(x, 4)) + 1           # SAI về nguyên tắc: dựa vào số thực

ks = [k for k in range(1, 40) for x in (4 ** k - 1, 4 ** k) if d_log(x) != d(x)]
print('k đầu tiên mà log số thực sai (x = 4^k hoặc 4^k - 1):', ks[:3])
x = 4 ** 24 - 1
print(f"x = 4^24 - 1: log = {math.log(x, 4)!r} -> d_log = {d_log(x)}, đúng = {d(x)}")
print('cách an toàn bằng bit_length:', (x.bit_length() + 1) // 2)

k đầu tiên mà log số thực sai (x = 4^k hoặc 4^k - 1): [24, 25, 26]
x = 4^24 - 1: log = 24.0 -> d_log = 25, đúng = 24
cách an toàn bằng bit_length: 24


**Vì sao sai:** với `x ≤ 10^9` của bài này `math.log` tình cờ vẫn cho đúng, nhưng với số lớn hơn (từ khoảng `4^24`) số thực làm tròn `4^k − 1` lên thành đúng `k` ⇒ `d` dư 1. Thói quen an toàn: dùng vòng chia số nguyên, `bit_length` (`d(x) = (x.bit_length() + 1) // 2`) hoặc so sánh với lũy thừa 4.

### Lỗi 4 — Duyệt từng số trong `[l, r]` (`10^9` phần tử)

In [20]:
t = time.time(); answer_linear([[1, 300000]]); tl = time.time() - t
print(f"duyệt 3·10^5 số: {tl:.2f}s -> 10^9 số sẽ mất hàng giờ; nhóm theo lũy thừa 4 chỉ ~16 bước")

duyệt 3·10^5 số: 0.08s -> 10^9 số sẽ mất hàng giờ; nhóm theo lũy thừa 4 chỉ ~16 bước


**Vì sao sai:** đúng nhưng quá chậm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lặp `x = ⌊x/b⌋` tới 0" ⇒ số chữ số cơ số `b`; nhóm các `x` có cùng số chữ số.
- "Mỗi thao tác xử lý `c` phần tử khác nhau" ⇒ đáp án thường `max(⌈S/c⌉, max w)`.
- Tổng một hàm "bậc thang" trên đoạn lớn ⇒ cộng theo bậc.
- Tránh `log` số thực khi cần số nguyên chính xác.

## 15. Biến thể của bài

1. Chia cho `b` bất kỳ ⇒ thay 4 bằng `b`.
2. Mỗi thao tác chọn `c` phần tử ⇒ `max(⌈S/c⌉, max w)`.
3. Mảng tùy ý (không liên tiếp) ⇒ cần `max(⌈S/2⌉, max d)` thật sự (max có thể lớn hơn).
4. Bài liên quan: LC 1953 *Maximum Number of Weeks for Which You Can Work*, LC 621 *Task Scheduler*.

In [21]:
# Biến thể 3: mảng tùy ý
def ops_any(nums):
    w = [d(x) for x in nums]
    return max((sum(w) + 1) // 2, max(w)) if len(w) > 1 else w[0]

for nums in ([1, 4 ** 10], [3, 5, 7], [64, 1, 1]):
    print(nums, ops_any(nums), '| tham lam:', greedy_pairs([d(x) for x in nums]))

[1, 1048576] 11 | tham lam: 11
[3, 5, 7] 3 | tham lam: 3
[64, 1, 1] 4 | tham lam: 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Khi nào dùng |
|---|---|---|---|
| 1 | BFS | mũ | Oracle nhỏ |
| 2 | Heap tham lam | `O(S log n)` | Kiểm chứng bổ đề |
| 3 | Công thức + duyệt | `O(r − l)` | Đoạn ngắn |
| 4 | Công thức + nhóm | `O(log r)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minOperations([[4, 5]])`.

**Bài 2 (tính tay):** `minOperations([[1, 16]])`.

**Bài 3 (điền TODO):** viết `digits_base(x, b)` (số chữ số cơ số `b`).

**Bài 4 (tự viết lại):** biến thể 2 — mỗi thao tác chọn `c` phần tử, chia cho `b`.

**Bài 5 (mini challenge):** viết lại `work_sum` bằng `F(r) − F(l−1)` và chứng minh bằng code rằng `⌈S/2⌉ ≥ d(r)` với mọi `1 ≤ l < r ≤ 2000`.

In [22]:
def digits_base_todo(x, b):
    # TODO
    raise NotImplementedError

def ops_c_b_todo(l, r, c, b):
    # TODO
    raise NotImplementedError

def check_bound_todo():
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `d(4) = d(5) = 2`. (2) `S = 4`. (3) `2`.

**Bài 2.** (1) Nhóm `1..3` (d=1), `4..15` (d=2), `16` (d=3). (2) `S = 3 + 24 + 3 = 30`. (3) `15`.

**Bài 3.** (1) Đếm số lần chia. (2) `x = 0` ⇒ 0 chữ số (theo nghĩa số lần chia). (3) So với `d` khi `b = 4`.

**Bài 4.** (1) Nhóm theo lũy thừa `b`. (2) `max(⌈S/c⌉, d(r))`. (3) Kiểm tra bằng heap chọn `c` phần tử lớn nhất.

**Bài 5.** (1) Dùng `F`. (2) Hai vòng `l, r`. (3) In số ca vi phạm (phải là 0).

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2, 3
a1 = Solution().minOperations([[4, 5]])
a2 = Solution().minOperations([[1, 16]])
assert a1 == 2 and a2 == 15 == answer_greedy([[1, 16]])

def digits_base(x, b):
    c = 0
    while x:
        x //= b; c += 1
    return c

assert all(digits_base(x, 4) == d(x) for x in range(1, 2000))
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: đúng')

Bài 1: 2 | Bài 2: 15 | Bài 3: đúng


In [24]:
# Bài 4: chọn c phần tử, chia cho b
def ops_c_b(l, r, c, b):
    S, k, lo = 0, 1, 1
    while lo <= r:
        a, e = max(l, lo), min(r, lo * b - 1)
        if a <= e:
            S += k * (e - a + 1)
        lo *= b; k += 1
    return max(-(-S // c), digits_base(r, b))

def greedy_c(work, c):
    h = [-w for w in work if w]
    heapq.heapify(h)
    ops = 0
    while h:
        taken = [-heapq.heappop(h) for _ in range(min(c, len(h)))]
        ops += 1
        for w in taken:
            if w > 1:
                heapq.heappush(h, -(w - 1))
    return ops

random.seed(4)
for _ in range(300):
    b, c = random.randint(2, 6), random.randint(2, 4)
    l = random.randint(1, 300); r = l + random.randint(c - 1, 60)
    assert ops_c_b(l, r, c, b) == greedy_c([digits_base(x, b) for x in range(l, r + 1)], c), (l, r, c, b)
print('Bài 4: khớp tham lam heap')

Bài 4: khớp tham lam heap


In [25]:
# Bài 5
viol = 0
for l in range(1, 2000):
    for r in range(l + 1, min(2001, l + 200)):
        S = F(r) - F(l - 1)
        assert S == work_sum(l, r)
        viol += (S + 1) // 2 < d(r)
print('Bài 5: số ca ⌈S/2⌉ < d(r):', viol)

Bài 5: số ca ⌈S/2⌉ < d(r): 0


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for (l, r): S = Σ_k k * |[l, r] ∩ [4^(k-1), 4^k - 1]|
            ans += max((S + 1) // 2, d(r))
```

**Checklist:**

- [ ] `d(x)` = số lần chia 4 tới 0 (không dùng `log` số thực)?
- [ ] Chia đôi **làm tròn lên**?
- [ ] Nhóm theo lũy thừa 4 (không duyệt từng số)?
- [ ] Cộng đáp án của **mọi** truy vấn?